## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr
import os

In [2]:
MODEL = "MiniMax-M2.7-highspeed"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [4]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, 
    model=MODEL,
    api_key=os.getenv("SUMOPOD_API_KEY"),
    base_url="https://ai.sumopod.com")

### These LangChain objects implement the method `invoke()`

In [5]:
retriever.invoke("Who is Avery?")

InvalidArgumentError: Collection expecting embedding with dimension of 3072, got 384

In [6]:
llm.invoke("Who is Avery?")

AIMessage(content='\n\nAvery is most often used as a given name (it works for both boys and girls).\u202fIt originated from an English surname that derived from the Old French\u202f*Alberi* or *Auberi*, which in turn came from the Germanic name\u202f*Alberic* meaning “noble”\u202f+\u202f“ruler” (often interpreted as “ruler of elves” or “wise ruler”).  \n\nBecause it’s a fairly common first name, there are many notable people who go by Avery—actors, musicians, athletes, businesspeople, etc.—as well as a few well‑known companies (e.g., Avery\u202fDennison, a global materials science company).  \n\nIf you’re asking about a specific Avery (e.g., a particular public figure, a character, or a brand), could you let me know which one you have in mind? That way I can give you a more targeted answer.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 966, 'prompt_tokens': 45, 'total_tokens': 1011, 'completion_tokens_details': {'accepted_prediction_toke

## Time to put this together!

In [7]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [8]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [9]:
answer_question("Who is Averi Lancaster?", [])

InvalidArgumentError: Collection expecting embedding with dimension of 3072, got 384

## What could possibly come next? 😂

In [ ]:
gr.ChatInterface(answer_question).launch()

/Users/danargh/Desktop/ai/llm_engineering/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/danargh/Desktop/ai/llm_engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/danargh/Desktop/ai/llm_engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/danargh/Desktop/ai/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/danargh/Desktop/ai/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1621, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/danargh/Desktop/ai/llm_engineering/.venv/lib/pyth

## Admit it - you thought RAG would be more complicated than that!!